In [1]:
import os
import sys
import time
import pickle
import warnings
from datetime import datetime, timedelta
from pathlib import Path
from difflib import SequenceMatcher
import numpy as np
import pandas as pd
import sklearn
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import NearestNeighbors
from scipy import stats
from scipy.spatial.distance import mahalanobis
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.ticker import MultipleLocator
import seaborn as sns
from statsmodels.distributions.empirical_distribution import ECDF
from pulp import LpMaximize, LpProblem, LpVariable, lpSum
import gurobipy as gp
from scipy.interpolate import interp1d
from gurobipy import Model, GRB
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from IPython.display import display
from preproces_prod3 import *

warnings.filterwarnings("ignore")

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

In [44]:
path_actual = Path.cwd()
path_data = path_actual.parent / 'Data'
path_nirsecl = path_actual.parent.parent/'Nirse_cl' / 'Data'
df_historic = (pd.read_csv(path_nirsecl/"data_sintrib.csv").drop(columns=['Unnamed: 0'])
                   .assign(fechaIng = lambda x: pd.to_datetime(x.fechaIng, format='%Y-%m-%d'), 
                           FECHA_NAC = lambda x: pd.to_datetime(x.FECHA_NAC, format='%Y-%m-%d'),
                           #VIVO = 'SI',
                           #FECHA_DEF = pd.NA,
                            ))
pf_all = (pd.read_csv(path_data / "pf_all.csv").assign(
        FECHA_NAC = lambda x: pd.to_datetime(x.FECHA_NAC, format = 'mixed'),
        year_nac = lambda x: x.FECHA_NAC.dt.year,
        month_nac = lambda x: x.FECHA_NAC.dt.month,
        elegibilidad_alt = lambda df: (df.year_nac + (df.month_nac >= 10).astype(int)),
        season = lambda x: np.where(x.month_nac.between(4, 9, inclusive='both'), 'in_season','pre_season'),
        ))

with open(path_data/'lista_ruts_cardio.pkl', 'rb') as f:
    lista_ruts_cardio = pickle.load(f)

In [46]:
df_historic.to_csv(path_data / "df_historic.csv", index=False)

In [45]:
criterio_pali_1 = (
pf_all
.assign(
    criterio_pali_normal = lambda x: ((x.RUN.isin(lista_ruts_cardio)) | (x.SEMANAS<32) | (x.PESO<1500)).astype(int),
    pali = lambda x: np.where(x.criterio_pali_normal==1, 1, 
                                np.where((x.year_nac==2023) & (x.month_nac.between(7, 9, inclusive='both')) & (x.SEMANAS<=34) & ((x.PESO<2500)), 1, 0))
        )
.query('(pali==1) | (SEMANAS<=35)')
.query('(24<=SEMANAS<=42) & (500<PESO<=p_99999_lognormal)')
.query('SEXO!=9')
.RUN
)

egresos_2024 = (
    df_historic
    [~df_historic.RUN.isin(df_historic.query('fechaIng < FECHA_NAC').RUN.unique())]
    .assign(fechaIng = lambda x: pd.to_datetime(x.fechaIng, format='%Y-%m-%d'),
            FECHA_NAC = lambda x: pd.to_datetime(x.FECHA_NAC, format='%Y-%m-%d'),
            ola_enfermedad=lambda df: df.fechaIng.dt.month.between(4, 9),
            elegibilidad_inyear=lambda df: np.where((df.elegibilidad == df.year) & (df.ola_enfermedad),'first_season',
                                                np.where(((df.elegibilidad + 2 == df.year) & (df.ola_enfermedad)) | 
                                                        ((df.elegibilidad + 1 == df.year) & (df.ola_enfermedad) & (df.season == "pre_season")) ,'second_season', #& (df.season == "in_season")
                                                        'discart')),
    # VIVO_mix = lambda df: np.where((df['VIVO']=='SI') | (df['VIVO'].isna()) & (df['FECHA_DEF'].isna()) , 'SI', 'NO')
    )
    [~df_historic.RUN.isin(df_historic[df_historic['fechaIng'] <= df_historic.FECHA_NAC + pd.DateOffset(days=7)].RUN.unique())]  
    .query('year==2024')
    .query('2022<=elegibilidad<=2024')
    .query('RUN.isin(@criterio_pali_1)')
    .sort_values(by='fechaIng')
    .drop_duplicates(subset=['RUN'], keep='first')
# .merge(pf_all[['RUN','season']],how='left',on='RUN')
    .query('ola_enfermedad')
    # .query('(VIVO_mix=="SI") | ((VIVO_mix=="NO") & (ingreso_mayor_1_mes>=7))')
   .groupby(['elegibilidad_inyear','year'])
   .VRS1.size()
   .unstack(0)
   .reset_index()

)
egresos_2024

elegibilidad_inyear,year,discart,first_season,second_season
0,2024,152,179,217


In [22]:
df_vrs_match_case = pd.read_csv(path_data/'df_vrs_match_case_02_06_25.csv')

df = (df_vrs_match_case
      .drop(columns=[col for col in df_vrs_match_case.columns if ((col.startswith('inm_')) | (col.startswith('MES_')) | (col.startswith('DIA_')) | 
                                               (col.startswith('SERC_')) | ((col.startswith('ANO_') & (col.endswith('TRAS')))) | 
                                               (col.startswith('AREAF')) | (col.startswith('dias_en')) | (col.startswith('fecha_tras')) |
                                               (col.endswith('Dall')) )])
      .assign(cardio1 = lambda x: x.RUN.isin(lista_ruts_cardio).astype(int),
              #displa = lambda x: x.RUN.isin(ruts_displasia.RUN).astype(int),
              fecha_nac = lambda x: pd.to_datetime(x.fecha_nac, infer_datetime_format=True),
              year_nac = lambda x: x.fecha_nac.dt.year,
              month_nac = lambda x: x.fecha_nac.dt.month,
              criterio_pali_normal = lambda x: ((x.RUN.isin(lista_ruts_cardio)) | (x.SEMANAS<32) | (x.PESO<1500)).astype(int),
              pali = lambda x: np.where(x.criterio_pali_normal==1, 1, 
                                        np.where((x.year_nac==2023) & (x.month_nac.between(7, 9, inclusive='both')) & (x.SEMANAS<=34) & ((x.PESO<2500)), 1, 0)))
      .copy()
      )

df_vrs_t2 = df.query('(pali==1) | (SEMANAS<=35)')

In [28]:
runes_179 = df_vrs_t2.query('diag_vrs==True').RUN.unique()

In [36]:
runes_raros = []
for run in runes_179:
    if run in df_historic.RUN.unique():
        continue
    else:
        runes_raros.append(run)

In [43]:
df_vrs_t2.query('RUN.isin(@runes_raros)')

,RUN,RUN_RNI,RUN_M,VACUNADO,MARCA,FECHA_NACIMIENTO,ANO_NAC,SEXO,SEMANAS,PESO,TALLA,EDAD_M,INS_C_M,INS_N_M,COMUNA,COMUNA_N,REG_RES,URBA_RURAL,NAC_MA,FECHA_INMUNIZACION,FECHA_DEFUNCION,CAUSA_DEFUNCION,VIVO,FALLECIDO_PREVIO,ESTAB,ServicioSalud,Seremi,P_ORIGEN,PREVI,FECHA_INGRESO,FECHA_EGRESO,AREA_FUNC_I,SER_CLIN_I,DIAS_ESTAD,COND_EGR,DIAG1,DIAG2,DIAG3,DIAG4,DIAG5,DIAG6,DIAG7,DIAG8,DIAG9,DIAG10,DIAG11,NOMBRE_REGION,Porcentaje Urbano,porcent_rural,percent_poor_multidim,percent_poor,p_00001_lognormal,p_99999_lognormal,fecha_nac,fechaIng_any,fechaEgr,fechaInm,VRS_D1,VRS_D1y3,diag_irag,diag_ira_alta,LRTI_Flag,LRTI_all_j,fechaIng_vrs,fechaIng_LRTI,group,sexo,prematuro_extremo,muy_prematuro,prematuro_moderado,prematuro,atypic_mom_age,region,Macrozona2,cama,fecha_upc,fecha_upc_vrs,days_upc,days_estad_vrs,is_rural,categori_macro,categori_regions,exp_rural,is_poor,vrs_pre_campaña,lrti_pre_campaña,any_pre_campaña,upc_pre_campaña,days_upc_vrs,event_upc,event,event_LRTI,event_any,take_nirse,cardio1,log_peso,Macrozones,super_preterm,ingreso_mayor_1_mes,VIVO_mix,fechaIng_vrs_copy,inmunizado,chile_chico,estado_inmunizacion,diag_vrs,diag_1_leter,edad_relativa,ingreso_relativo,year_nac,month_nac,criterio_pali_normal,pali
143351,db97ccd245df4a064b20e703623e43804e14dce53f3bb5befb947e0803dd6fd0,1.0,b5809794edaedab3716a979672b9e77f0a1a2a403899aec11d32a957541dcdef,SI,0.0,09Feb2024,2024.0,1.0,39.0,3880.0,51.0,37.0,5.0,1.0,2101.0,2101.0,2.0,0.0,C,2024-04-09,NaN,NaN,SI,VIVO,112200.0,NaN,13.0,152.0,2.0,21May2024,28May2024,0,NaN,7.0,1.0,J219,NaN,B978,J960,Q211,Q210,K522,NaN,NaN,NaN,NaN,Región De Antofagasta,0.985667,0.014333,0.166543,7.232800,1994.452036,5640.163672,2024-02-09,2024-05-21,2024-05-28,2024-04-09,1,1,True,False,True,True,2024-05-21,2024-05-21,CATCH_UP,1,0,0,0,0,0,ANTOFAGASTA,Norte,NaN,NaN,NaN,0,7.0,0,2,2,1.014437,0,0,0,0,0,0,0,1,1,1,True,1,8.263590,North Macrozone,0,102.0,SI,2024-05-21,1,0,1,1,J,104,50.0,2024,2,1,1
143556,ad9e5f32c7e34b52e22b5778b45bd95476c79bed20c550712e75fb43e409085c,1.0,366aeac2e056bfa5ebc31001a83ff6780f1611256f5a669a239f5d778261acf2,SI,0.0,23Dec2023,2023.0,1.0,30.0,1165.0,38.0,29.0,5.0,1.0,13114.0,13114.0,13.0,0.0,C,2024-04-18,NaN,NaN,SI,VIVO,112200.0,NaN,13.0,152.0,2.0,24Jun2024,25Jun2024,0,NaN,1.0,1.0,J219,NaN,J981,J960,P271,NaN,NaN,NaN,NaN,NaN,NaN,Región Metropolitana de Santiago,1.000000,0.000000,0.044322,0.909325,918.378723,2492.476304,2023-12-23,2024-06-24,2024-06-25,2024-04-18,1,1,True,False,True,True,2024-06-24,2024-06-24,CATCH_UP,1,0,1,0,1,0,METROPOLITANA,Centro,NaN,NaN,NaN,0,1.0,0,1,12,1.000000,0,0,0,0,0,0,0,1,1,1,True,0,7.060476,Central Macrozone,1,184.0,SI,2024-06-24,1,1,1,1,J,56,84.0,2023,12,1,1


In [ ]:
df_historic